## Google Colab

In [5]:
!pip install faiss-cpu peft sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 164.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 47.5 MB/s eta 0:00:00


In [6]:
import sys
import os

In [7]:
!git clone https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git
%cd Emasters_Group-2_CapstoneProject
sys.path.append('/content/Emasters_Group-2_CapstoneProject')
!ls -la /content/Emasters_Group-2_CapstoneProject
print("Current Working Directory:", os.getcwd())

Cloning into 'Emasters_Group-2_CapstoneProject'...
remote: Enumerating objects: 528, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 528 (delta 120), reused 49 (delta 33), pack-reused 310 (from 1)
Receiving objects: 100% (528/528), 14.77 MiB | 46.98 MiB/s, done.
Resolving deltas: 100% (253/253), done.
/content/Emasters_Group-2_CapstoneProject/Emasters_Group-2_CapstoneProject
total 204
drwxr-xr-x 11 root root  4096 Aug  5 10:41 .
drwxr-xr-x  1 root root  4096 Aug  5 10:39 ..
-rw-r--r--  1 root root 94795 Aug  5 10:39 check_point_1.ipynb
-rw-r--r--  1 root root 11597 Aug  5 10:39 checkpoint_2.ipynb
drwxr-xr-x  2 root root  4096 Aug  5 10:39 configs
drwxr-xr-x  4 root root  4096 Aug  5 10:39 data
drwxr-xr-x  2 root root  4096 Aug  5 10:39 Emasters-Codegen-Project-IIITH-Presentation
drwxr-xr-x 10 root root  4096 Aug  5 10:41 Emasters_Group-2_CapstoneProject
drwxr-xr-x  8 root root  4096 Aug  5 10:39 .git
drwxr-xr-x  3 r

In [8]:
sys.path.append('/content/Emasters_Group-2_CapstoneProject')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/indexing')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/generators')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/evaluation')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/rag')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/embeddings')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/configs')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/output')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/outputs')

In [12]:
from pathlib import Path
from src import config
import json
import json as _json
sys.path.append(str(Path.cwd()))
import os
import yaml
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from src.data.data_loader import DatasetLoader
from src.models.load_model import ModelLoader
from src.generators.program_generator import GenerationPipeline
from src.generators.doc_gen import DocGenerator
from src.generators.text_to_sql import TextToSQLGenerator
from src.generators.commit_gen import CommitMessageGenerator
from src.evaluation.comparator import ModelComparator
from src.evaluation.visualization import ResultVisualizer
from src.rag.rag_pipeline import RAGPipeline
from src.embeddings.ast_indexing import ASTIndexer
from src.embeddings.embedding import CodeEmbedder
from src.embeddings.indexing import SemanticIndexManager
import pickle
import re
from src.evaluation.metrics import EvaluationMetrics

In [13]:
# -------------------------------------------------------------
# 1. Config Parser Setup
# -------------------------------------------------------------
class DictToObject:
    def __init__(self, data: dict):
        for key, value in data.items():
            if isinstance(value, dict):
                setattr(self, key, DictToObject(value))
            else:
                setattr(self, key, value)

def load_config(yaml_path: str = "configs/config.yaml") -> DictToObject:
    path = Path(yaml_path)
    if not path.exists():
        raise FileNotFoundError(f"Config file not found at: {path.resolve()}")

    with open(path, "r", encoding="utf-8") as f:
        config_dict = yaml.safe_load(f) or {}

    return DictToObject(config_dict)

config = load_config("configs/config.yaml")

print(f"Base Model: {config.model_name}")
print(f"Data Directory: {config.data_paths.raw_dir}")

Base Model: Qwen/Qwen2.5-Coder-7B-Instruct
Data Directory: data/raw


In [14]:
# -------------------------------------------------------------
# 2. Data Loader Step
# -------------------------------------------------------------
print("\n--- Loading Datasets ---")

data_dir = Path(config.data_paths.raw_dir)
data_dir.mkdir(parents=True, exist_ok=True)

benchmark_placeholders = {
    "Spider": config.data_paths.spider
}

for folder_name, placeholder_filename in benchmark_placeholders.items():
    folder_path = data_dir / folder_name
    folder_path.mkdir(parents=True, exist_ok=True)

    has_data = any(folder_path.glob("*.json")) or any(folder_path.glob("*.jsonl"))
    if not has_data:
        placeholder_path = folder_path / placeholder_filename
        print(f"Creating placeholder for empty benchmark folder: {placeholder_path}")
        with open(placeholder_path, "w", encoding="utf-8") as f:
            f.write('{"question": "Sample Query", "query": "SELECT * FROM sample;"}\n')

# Initialize loader
data_loader = DatasetLoader(data_dir="data/raw")

# Loads all .json/.jsonl files inside data/raw/Spider/
spider_data = data_loader.load_spider()

# Combine into a single corpus
code_corpus = spider_data # + bird_data + codoc_data
print(f"Total corpus samples loaded: {len(code_corpus)}")


--- Loading Datasets ---
Total corpus samples loaded: 12389


In [15]:
# -------------------------------------------------------------
# 3. LoRA Fine-Tuning Setup
# -------------------------------------------------------------
print("\n--- Fine-Tuning with LoRA ---")
model_loader = ModelLoader(config)
tokenizer = model_loader.load_tokenizer()
base_model = model_loader.load_base_model()

lora_model = model_loader.setup_lora_training(
    base_model,
    r=config.lora.r,
    alpha=config.lora.alpha
)

# Build a small training set from the processed task files
def _load_jsonl(path):
    items = []
    p = Path(path)
    if not p.exists():
        return items
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

train_texts = []  # list of (prompt, completion) tuples

def _completion_prompt(instruction):
    return f"User Question: {instruction}\n\nAnswer: "

for item in _load_jsonl("data/processed/program_generation.jsonl"):
    instr, code = item.get("instruction") or item.get("prompt", ""), item.get("code", "")
    if instr and code:
        train_texts.append((_completion_prompt(instr), code))

for item in _load_jsonl("data/processed/doc_generation.jsonl"):
    instr = item.get("instruction") or item.get("prompt", "")
    out = item.get("docstring") or item.get("output", "")
    if instr and out:
        train_texts.append((_completion_prompt(instr), out))

for item in _load_jsonl("data/processed/commit_generation.jsonl"):
    instr = item.get("instruction") or item.get("prompt", "")
    out = item.get("commit_message") or item.get("output", "")
    if instr and out:
        train_texts.append((_completion_prompt(instr), out))

print(f"Prepared {len(train_texts)} training examples for LoRA fine-tuning.")

lora_model = model_loader.train_lora(
    lora_model,
    tokenizer,
    train_texts,
    epochs=3,
    batch_size=1,
    grad_accum_steps=4,
    learning_rate=1e-4,
    max_length=256,
)

lora_checkpoint_dir = Path(config.lora.output_dir)
lora_checkpoint_dir.mkdir(parents=True, exist_ok=True)
lora_model.save_pretrained(str(lora_checkpoint_dir))
tokenizer.save_pretrained(str(lora_checkpoint_dir))
print(f"LoRA fine-tuned adapter saved to: {lora_checkpoint_dir}")

models, tokenizer = model_loader.load_models(lora_path=str(lora_checkpoint_dir))


--- Fine-Tuning with LoRA ---


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
LoRA fine-tuned adapter saved to: models/checkpoints/lora_finetuned


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [16]:
# -------------------------------------------------------------
# 4. Build FAISS Semantic Index & AST Index
# -------------------------------------------------------------
print("\n--- Building Semantic & AST Indices ---")

# 1. Initialize Embedder
print("[1/4] Loading embedding model...")
embedder = CodeEmbedder(
    model_name=config.embedding_model
)

# 2. Generate embeddings for the entire code corpus
print(f"[2/4] Generating embeddings for {len(code_corpus)} samples...")
corpus_embeddings = embedder.generate_embedding(
    data=code_corpus,
    is_query=False,
    normalize=True,
    batch_size=32
)
print(f"✅ Generated embeddings with shape: {corpus_embeddings.shape}")

# 3. Build FAISS semantic index
print("[3/4] Building FAISS semantic index...")
embedding_dim = corpus_embeddings.shape[1]
semantic_index = SemanticIndexManager(
    embedding_dim=embedding_dim,
    index_type="Flat"
)
semantic_index.add_codes(corpus_embeddings, code_corpus)

# Save FAISS index
faiss_index_path = Path(config.indices_paths.faiss_index)
faiss_index_path.parent.mkdir(parents=True, exist_ok=True)
semantic_index.save(faiss_index_path)
print(f"✅ FAISS index saved to: {faiss_index_path}")

# 4. Build AST index
print("[4/4] Building AST index...")
ast_indexer = ASTIndexer(language="python")
ast_store = []

def looks_like_python_code(text: str) -> bool:
    """Check if text looks like Python code (has def/class/import keywords)."""
    if not text or len(text.strip()) < 10:
        return False
    python_indicators = ['def ', 'class ', 'import ', 'from ', 'return ', 'if ', 'for ', 'while ']
    return any(indicator in text for indicator in python_indicators)

# Only process items that actually contain Python code
for idx, item in enumerate(code_corpus[:500]):  # Limit to first 500 for speed
    code_text = ""

    if isinstance(item, dict):
        # CoDocBench: has 'version_data' with actual Python code
        if 'version_data' in item and isinstance(item['version_data'], list):
            if len(item['version_data']) > 0 and 'code' in item['version_data'][0]:
                code_text = item['version_data'][0]['code']
        # Generic fallback for other datasets with code
        elif 'code' in item and isinstance(item['code'], str):
            code_text = item['code']

    # Only attempt AST parsing if it looks like Python code
    if code_text and looks_like_python_code(code_text):
        ast_structure = ast_indexer.parse_structure(code_text)
        if ast_structure.get('status') == 'success':
            ast_store.append({
                "index": idx,
                "structure": ast_structure,
                "code_preview": code_text[:100]
            })

# Save AST index
ast_index_path = Path(config.indices_paths.ast_store)
ast_index_path.parent.mkdir(parents=True, exist_ok=True)
with open(ast_index_path, "wb") as f:
    pickle.dump(ast_store, f)
print(f"✅ AST index saved to: {ast_index_path}")
print(f" Indexed {len(ast_store)} Python code samples (skipped non-Python text)")

# 5. Test semantic search
print("\n--- Testing Semantic Search ---")

# Reset model to eval mode and clear any cached state
embedder.model.eval()
if hasattr(embedder.model, 'reset_cache'):
    embedder.model.reset_cache()

test_query = "Calculate the area of a circle"
print(f"Query: '{test_query}'")

# Add timing to identify bottlenecks
import time
start_time = time.time()

try:
    print("Generating query embedding...")
    query_embedding = embedder.generate_embedding(
        data=[test_query],  # Pass as list for consistency
        is_query=True,
        normalize=True,
        batch_size=1  # Single item batch
    )
    print(f"✅ Query embedding generated in {time.time() - start_time:.2f}s")
    print(f"   Shape: {query_embedding.shape}")

    print("Searching FAISS index...")
    search_start = time.time()
    search_results = semantic_index.search(query_embedding, k=3)
    print(f"✅ Search completed in {time.time() - search_start:.2f}s")

    print(f"\nTop {len(search_results)} results:")
    for i, (metadata, score) in enumerate(search_results, 1):
        code_preview = ""
        if isinstance(metadata, dict):
            if 'instruction' in metadata:
                code_preview = metadata['instruction'][:80]
            elif 'question' in metadata:
                code_preview = metadata['question'][:80]
            elif 'code' in metadata:
                code_preview = metadata['code'][:80]
            else:
                code_preview = str(metadata)[:80]
        print(f"  {i}. Score: {score:.4f} | Preview: {code_preview}...")

except Exception as e:
    print(f"❌ Error during semantic search test: {e}")
    import traceback
    traceback.print_exc()

print(f"\n✅ Indexing pipeline completed successfully!")


--- Building Semantic & AST Indices ---
[1/4] Loading embedding model...


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[2/4] Generating embeddings for 12389 samples...
✅ Generated embeddings with shape: (12389, 384)
[3/4] Building FAISS semantic index...
✅ FAISS index saved to: data/indices/faiss_semantic.index
[4/4] Building AST index...
✅ AST index saved to: data/indices/ast_store.pkl
 Indexed 0 Python code samples (skipped non-Python text)

--- Testing Semantic Search ---
Query: 'Calculate the area of a circle'
Generating query embedding...
✅ Query embedding generated in 0.01s
   Shape: (1, 384)
Searching FAISS index...
✅ Search completed in 0.12s

Top 3 results:
  1. Score: 0.6671 | Preview: {'file': 'skimage__shared__geometry.py', 'function': 'polygon_area', 'version_da...
  2. Score: 0.6671 | Preview: {'file': 'skimage__shared__geometry.py', 'function': 'polygon_area', 'version_da...
  3. Score: 0.6459 | Preview: {'file': 'skimage_draw_draw.py', 'function': 'circle', 'version_data': [{'commit...

✅ Indexing pipeline completed successfully!


In [17]:
# -------------------------------------------------------------
# 5. Specialized Task Generation Modules
# -------------------------------------------------------------
print("\n--- Running Downstream Tasks ---")
base_gen_pipeline = GenerationPipeline(models["base"], tokenizer, config.generation)
lora_gen_pipeline = GenerationPipeline(
    models["lora"] if models["lora"] else models["base"],
    tokenizer,
    config.generation,
)

# Initialize RAG Pipeline
rag_pipeline = RAGPipeline(
    model=models["lora"] if models["lora"] else models["base"],
    tokenizer=tokenizer,
    embedder=embedder,
    index_manager=semantic_index,
    config=config.generation
)

# Task 1: Documentation Generation
doc_gen = DocGenerator(lora_gen_pipeline)
docstring = doc_gen.generate_docstring(str(code_corpus[0])[:500])
print(f"\n[Task: Doc Generation]\nOutput:\n{docstring[:500]}...")

# Task 2: Text-to-SQL
text_to_db = TextToSQLGenerator(pipeline=lora_gen_pipeline)
res = text_to_db.generate_queries(
    question="Find the total number of employees in the engineering department who joined after 2022.",
    schema="Table employee(id, name, department, join_year)",
    dialect="sqlite",
)
print(f"\n[Task: Text-to-SQL]\n{json.dumps(res, indent=2)}")

saved_path = text_to_db.save_result(
    res,
    output_dir="outputs/generated/text_to_sql",
    question="Find the total number of employees in the engineering department who joined after 2022.",
    schema="Table employee(id, name, department, join_year)",
    dialect="sqlite",
)
print(f"Text-to-SQL result saved to: {saved_path}")

# Task 3: RAG-Enhanced Code Generation
print("\n[Task: RAG-Enhanced Code Generation]")
rag_query = "How do I calculate the average of a list of numbers in Python?"
rag_output, retrieved_context = rag_pipeline.generate_with_rag(rag_query, top_k=2)
print(f"Query: {rag_query}")
print(f"Retrieved {len(retrieved_context)} relevant code snippets")
print(f"Generated Answer:\n{rag_output[:500]}...")

# Task 4: Commit Message Generation
commit_gen = CommitMessageGenerator(lora_gen_pipeline)
commit_msg = commit_gen.generate_commit_msg("diff --git a/main.ipynb b/main.ipynb\n+ import os")
print(f"\n[Task: Commit Msg Generation]\nOutput:\n{commit_msg}")

print("\n--- Running Baseline & LoRA Inference for Evaluation ---")

eval_query = "Write a Python function named `circle_area` that calculates the area of the circle given its radius."
eval_corpus = code_corpus[:1]

prompt = f"User Question: {eval_query}\n\nAnswer: "
lora_model_ref = models.get("lora") or models["base"]

print("Generating base vs. LoRA-adapted responses from the trained model...")
base_output, lora_output = model_loader.generate_base_vs_lora(
    lora_model_ref,
    tokenizer,
    prompt,
    max_new_tokens=256,
    temperature=0.2,
)

print("✅ Base and LoRA outputs ready for comparison!")


--- Running Downstream Tasks ---

[Task: Doc Generation]
Output:
```python
def get_attribution_method(instance_id: str, instruction: str, type: str) -> dict:
    """
    Returns a dictionary containing the instance ID, instruction, and type.

    Args:
        instance_id (str): The unique identifier for the instance.
        instruction (str): A description of the task or request.
        type (str): The category or type of the request.

    Returns:
        dict: A dictionary containing the instance ID, instruction, and type.
    """
```

Google Style Docst...

[Task: Text-to-SQL]
{
  "sql_query": "SELECT COUNT(*) FROM employee WHERE department = 'engineering' AND join_year > 2022",
  "mongodb_query": {
    "collection": "employee",
    "operation": "countDocuments",
    "filter": {
      "department": "engineering",
      "join_year": {
        "$gt": 2022
      }
    }
  }
}
Text-to-SQL result saved to: outputs/generated/text_to_sql/text_to_sql_20260805_112820.json

[Task: RAG-Enh

In [18]:
# -------------------------------------------------------------
# 6. Comprehensive Evaluation, Comparison & Visualization
# -------------------------------------------------------------
print("\n--- Calculating Comprehensive Evaluation Metrics ---")

comparator = ModelComparator(device=config.evaluation.device if hasattr(config, "evaluation") else "cpu")

test_assertions = [
    "import math\n\nassert math.isclose(circle_area(2), 12.566370614359172, rel_tol=1e-5)"
]

def _load_jsonl(path):
    items = []
    p = Path(path)
    if not p.exists():
        return items
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

program_gen_data = _load_jsonl("data/processed/program_generation.jsonl")
circle_area_ref = next(
    (item["code"] for item in program_gen_data if "circle_area" in item.get("code", "")),
    "import math\n\ndef circle_area(radius):\n    return math.pi * radius ** 2",
)

eval_results = comparator.compare(
    references=[circle_area_ref],
    base_preds=[base_output],
    lora_preds=[lora_output],
    rag_preds=[],
    test_cases=test_assertions,
)

print("\nPipeline Comparison Metrics:")
for model_type, metrics in eval_results.items():
    print(f"{model_type:15s} -> {metrics}")

# =====================================================================
# SQL & DBT Task Evaluation
# =====================================================================
print("\n--- Evaluating SQL Generation Quality ---")

sql_dbt_tasks = _load_jsonl("data/processed/spider_eval.jsonl")
print(f"Found {len(sql_dbt_tasks)} properly-schema'd SQL samples.")

eval_samples = sql_dbt_tasks[:5] if len(sql_dbt_tasks) >= 5 else sql_dbt_tasks

generated_sqls, gold_sqls, db_ids, instructions = [], [], [], []

for idx, item in enumerate(eval_samples):
    instance_id = item.get('id', f'task_{idx}')
    question = item.get('question', '')
    schema = item.get('schema', 'N/A')
    db_id = item.get('db_id', '')

    if not question:
        continue

    print(f"\n[{len(generated_sqls)+1}/{len(eval_samples)}] Processing: {instance_id}")
    print(f"    Prompt: {question[:80]}...")

    gold_sql = item.get('gold_sql', '')

    result = text_to_db.generate_queries(question=question, schema=schema, dialect="sqlite")
    generated_sql = result.get('sql_query', '')
    generated_sqls.append(generated_sql)
    db_ids.append(db_id)

    if gold_sql:
        gold_sqls.append(gold_sql)
        instructions.append("")
        print(f"    ✅ Gold SQL found. Generated: {generated_sql[:60]}...")
    else:
        gold_sqls.append("")
        instructions.append(question)

valid_gold_pairs = [(g, p, d) for g, p, d in zip(gold_sqls, generated_sqls, db_ids) if g and p]
valid_instruct_pairs = [(i, p) for i, p, g in zip(instructions, generated_sqls, gold_sqls) if i and not g and p]

metrics_summary = {}

if valid_gold_pairs:
    gold_only = [pair[0] for pair in valid_gold_pairs]
    pred_only = [pair[1] for pair in valid_gold_pairs]
    dbid_only = [pair[2] for pair in valid_gold_pairs]

    exact_matches = sum(1 for g, p in zip(gold_only, pred_only)
                        if re.sub(r'\s+', ' ', g.strip().lower().rstrip(';')) == re.sub(r'\s+', ' ', p.strip().lower().rstrip(';')))
    em_acc = exact_matches / len(gold_only)

    db_paths = [
        str(Path("data/raw/Spider/spider_data/database") / db_id / f"{db_id}.sqlite")
        for db_id in dbid_only
    ]
    exec_acc = EvaluationMetrics.compute_sql_execution_accuracy(gold_only, pred_only, db_paths)

    bleu_score = EvaluationMetrics.compute_bleu(gold_only, pred_only)
    bert_score = EvaluationMetrics.compute_bertscore(gold_only, pred_only, device="cpu")

    metrics_summary['SQL_Exact_Match'] = round(em_acc, 4)
    metrics_summary['SQL_Execution_Accuracy'] = round(exec_acc, 4)
    metrics_summary['SQL_BLEU'] = round(bleu_score, 4)
    metrics_summary['SQL_CodeBERTScore'] = round(bert_score, 4)

    print(f"\n✅ Standard Text-to-SQL Evaluation Results ({len(gold_only)} samples):")
    print(f"   Exact Match Accuracy:      {em_acc * 100:.2f}%")
    print(f"   Execution Accuracy:        {exec_acc * 100:.2f}%")
    print(f"   BLEU Score:                {bleu_score:.4f}")
    print(f"   CodeBERTScore:             {bert_score:.4f}")

if metrics_summary:
    for key, value in metrics_summary.items():
        eval_results["LoRA_Model"][key] = value

visualizer = ResultVisualizer(output_dir=config.outputs.plots_dir)
visualizer.plot_comparison(eval_results, save_name="model_comparison_checkpoint2.png")

print("\n✅ pipeline execution completed successfully!")


--- Calculating Comprehensive Evaluation Metrics ---


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            


Pipeline Comparison Metrics:
Base_Model      -> {'BLEU': 0.0162, 'CodeBERTScore': 0.9488, 'F1_Score': 0.0901, 'ROUGE-1': 0.1224, 'ROUGE-L': 0.1088, 'Execution_Accuracy': 1.0}
LoRA_Model      -> {'BLEU': 0.0352, 'CodeBERTScore': 0.9613, 'F1_Score': 0.0597, 'ROUGE-1': 0.122, 'ROUGE-L': 0.122, 'Execution_Accuracy': 1.0}

--- Evaluating SQL Generation Quality ---
Found 3243 SQL/DBT-related samples out of 12389 total.

[1/5] Processing: playbook001
    Prompt: Complete the project of this database to show the metrics of each traffic source...
    ✅ Generated valid SQL structure: SELECT traffic_source, COUNT(*) AS total_visits, SUM(metric_...

[2/5] Processing: provider001
    Prompt: How can I map Medicare specialties to NUCC taxonomy codes, prioritize the most s...
    ✅ Generated valid SQL structure: SELECT DISTINCT T2.nuucc_code AS primary_taxonomy_code, T1.p...

[3/5] Processing: asana001
    Prompt: Can you describe the process used to aggregate task and project metrics for Asan...
  